In [28]:
# ============================================================
# CELL 1 — PROJECT CONFIGURATION & DIRECTORY SETUP
# ============================================================

from pathlib import Path
import json
import os
import gc

# ------------------------------------------------------------
# 1. Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# ------------------------------------------------------------
# 2. Directory structure
# ------------------------------------------------------------

DIRS = {
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "data_chunks": PROJECT_ROOT / "data" / "chunks",
    "embeddings": PROJECT_ROOT / "embeddings",
    "vectorstore": PROJECT_ROOT / "vectorstore",
    "evaluation": PROJECT_ROOT / "evaluation",
    "results": PROJECT_ROOT / "results",
    "logs": PROJECT_ROOT / "logs",
    "checkpoints": PROJECT_ROOT / "checkpoints",
}

# Create directories safely
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Dataset configuration
# ------------------------------------------------------------

DATASET_FILENAME = "synthetic_knowledge_items.csv"

DATASET_PATH = Path("E:/rag/synthetic_knowledge_items.csv")

# ------------------------------------------------------------
# 4. RAG baseline configuration
# ------------------------------------------------------------

CONFIG = {
    "project": {
        "name": "Synthetic IT Knowledge RAG",
        "dataset_name": "Synthetic IT-Related Knowledge Items",
        "dataset_version": "Version 3",
        "expected_records": 100,
        "expected_columns": 4,
    },

    "dataset": {
        "filename": DATASET_FILENAME,
        "raw_path": str(DATASET_PATH),
    },

    "chunking": {
        "chunk_size": 500,
        "chunk_overlap": 50,
    },

    "retrieval": {
        "method": "dense",
        "top_k": 5,
    },

    "generation": {
        "provider": "ollama",
        "base_url": "http://localhost:11434",
        "model": None,  # Set after checking installed Ollama models
    },
}

# ------------------------------------------------------------
# 5. Persistent configuration checkpoint
# ------------------------------------------------------------

CONFIG_CHECKPOINT = DIRS["checkpoints"] / "project_config.json"

try:
    with open(CONFIG_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(CONFIG, f, indent=4)

    print("[SUCCESS] Project configuration saved.")
    
except Exception as e:
    print("[ERROR] Failed to save project configuration.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise

# ------------------------------------------------------------
# 6. Environment summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RAG PROJECT INITIALIZATION")
print("=" * 60)

print(f"Project root       : {PROJECT_ROOT}")
print(f"Dataset path       : {DATASET_PATH}")
print(f"Dataset expected   : {CONFIG['project']['expected_records']} records")
print(f"Expected columns   : {CONFIG['project']['expected_columns']}")
print(f"Chunk size         : {CONFIG['chunking']['chunk_size']}")
print(f"Chunk overlap      : {CONFIG['chunking']['chunk_overlap']}")
print(f"Retrieval method   : {CONFIG['retrieval']['method']}")
print(f"Top-K              : {CONFIG['retrieval']['top_k']}")
print(f"Ollama URL         : {CONFIG['generation']['base_url']}")

print("=" * 60)
print("[SUCCESS] Initialization completed.")

[SUCCESS] Project configuration saved.

RAG PROJECT INITIALIZATION
Project root       : e:\rag
Dataset path       : E:\rag\synthetic_knowledge_items.csv
Dataset expected   : 100 records
Expected columns   : 4
Chunk size         : 500
Chunk overlap      : 50
Retrieval method   : dense
Top-K              : 5
Ollama URL         : http://localhost:11434
[SUCCESS] Initialization completed.


In [29]:
# ============================================================
# CELL 2 — OLLAMA CONNECTION & MODEL DISCOVERY
# ============================================================

import requests
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Ollama configuration
# ------------------------------------------------------------

OLLAMA_BASE_URL = CONFIG["generation"]["base_url"]
OLLAMA_TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"

OLLAMA_CHECKPOINT = (
    DIRS["checkpoints"] / "ollama_environment.json"
)

# ------------------------------------------------------------
# 2. Check Ollama server
# ------------------------------------------------------------

print("=" * 60)
print("OLLAMA CONNECTION CHECK")
print("=" * 60)

try:
    response = requests.get(
        OLLAMA_TAGS_URL,
        timeout=10
    )

    response.raise_for_status()

    ollama_data = response.json()

    print("[SUCCESS] Ollama server is running.")
    print(f"[INFO] Ollama URL: {OLLAMA_BASE_URL}")

except requests.exceptions.ConnectionError:
    print("[ERROR] Could not connect to Ollama.")
    print(f"[ERROR] Expected Ollama at: {OLLAMA_BASE_URL}")
    print()
    print("Make sure Ollama is running on your computer.")
    raise

except requests.exceptions.Timeout:
    print("[ERROR] Ollama connection timed out.")
    print(f"[ERROR] URL: {OLLAMA_BASE_URL}")
    raise

except requests.exceptions.HTTPError as e:
    print("[ERROR] Ollama returned an HTTP error.")
    print(f"[ERROR] Status code: {response.status_code}")
    print(f"[ERROR] Details: {e}")
    raise

except Exception as e:
    print("[ERROR] Unexpected Ollama connection error.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

# ------------------------------------------------------------
# 3. Extract installed models
# ------------------------------------------------------------

try:
    installed_models = ollama_data.get("models", [])

    if not installed_models:
        raise RuntimeError(
            "Ollama is running, but no local models were found."
        )

    model_names = [
        model.get("name")
        for model in installed_models
        if model.get("name")
    ]

    print(f"\n[INFO] Installed Ollama models: {len(model_names)}")

    for index, model_name in enumerate(model_names, start=1):
        print(f"  {index}. {model_name}")

except Exception as e:
    print("[ERROR] Failed to inspect Ollama models.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

# ------------------------------------------------------------
# 4. Display model information
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("MODEL INFORMATION")
print("-" * 60)

for model in installed_models:
    name = model.get("name", "Unknown")
    size = model.get("size", 0)

    # Convert bytes → GB
    size_gb = size / (1024 ** 3) if size else 0

    print(f"Model : {name}")
    print(f"Size  : {size_gb:.2f} GB")
    print()

# ------------------------------------------------------------
# 5. Save Ollama environment checkpoint
# ------------------------------------------------------------

ollama_environment = {
    "base_url": OLLAMA_BASE_URL,
    "server_status": "connected",
    "installed_models": model_names,
    "model_details": installed_models,
}

try:
    with open(
        OLLAMA_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            ollama_environment,
            f,
            indent=4
        )

    print("[SUCCESS] Ollama environment checkpoint saved.")
    print(f"[INFO] Checkpoint: {OLLAMA_CHECKPOINT}")

except Exception as e:
    print("[ERROR] Failed to save Ollama checkpoint.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

print("=" * 60)
print("[SUCCESS] Ollama environment is ready.")
print("=" * 60)

OLLAMA CONNECTION CHECK
[SUCCESS] Ollama server is running.
[INFO] Ollama URL: http://localhost:11434

[INFO] Installed Ollama models: 5
  1. qwen3:8b
  2. qwen3.8:27b-mtp-q4_K_M
  3. qwen3.8:27b
  4. gemma4:12b
  5. gemma4:26b

------------------------------------------------------------
MODEL INFORMATION
------------------------------------------------------------
Model : qwen3:8b
Size  : 4.87 GB

Model : qwen3.8:27b-mtp-q4_K_M
Size  : 16.52 GB

Model : qwen3.8:27b
Size  : 16.52 GB

Model : gemma4:12b
Size  : 7.04 GB

Model : gemma4:26b
Size  : 16.75 GB

[SUCCESS] Ollama environment checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\ollama_environment.json
[SUCCESS] Ollama environment is ready.


In [30]:
# ============================================================
# CELL 3 — DATASET LOADING & INITIAL VALIDATION
# ============================================================

import pandas as pd
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Verify dataset path
# ------------------------------------------------------------

print("=" * 60)
print("DATASET LOADING & VALIDATION")
print("=" * 60)

try:
    if not DATASET_PATH.exists():
        raise FileNotFoundError(
            f"Dataset not found at: {DATASET_PATH}"
        )

    if not DATASET_PATH.is_file():
        raise FileNotFoundError(
            f"Dataset path is not a file: {DATASET_PATH}"
        )

    print(f"[SUCCESS] Dataset found:")
    print(f"         {DATASET_PATH}")

except Exception as e:
    print("[ERROR] Dataset verification failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------

try:
    df_raw = pd.read_csv(DATASET_PATH)

    if df_raw.empty:
        raise ValueError("Dataset is empty.")

    print("\n[SUCCESS] Dataset loaded successfully.")

except Exception as e:
    print("[ERROR] Failed to load dataset.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 3. Basic dimensions
# ------------------------------------------------------------

try:
    n_rows, n_columns = df_raw.shape

    print("\n" + "-" * 60)
    print("DATASET DIMENSIONS")
    print("-" * 60)

    print(f"Rows    : {n_rows:,}")
    print(f"Columns : {n_columns}")

except Exception as e:
    print("[ERROR] Failed to inspect dataset dimensions.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Validate expected dimensions
# ------------------------------------------------------------

EXPECTED_ROWS = CONFIG["project"]["expected_records"]
EXPECTED_COLUMNS = CONFIG["project"]["expected_columns"]

if n_rows != EXPECTED_ROWS:
    print(
        f"[WARNING] Expected {EXPECTED_ROWS} rows "
        f"but found {n_rows}."
    )
else:
    print(f"[SUCCESS] Record count matches expected value: {EXPECTED_ROWS}")

if n_columns != EXPECTED_COLUMNS:
    print(
        f"[WARNING] Expected {EXPECTED_COLUMNS} columns "
        f"but found {n_columns}."
    )
else:
    print(
        f"[SUCCESS] Column count matches expected value: "
        f"{EXPECTED_COLUMNS}"
    )


# ------------------------------------------------------------
# 5. Exact column names
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("COLUMN INFORMATION")
print("-" * 60)

for index, column in enumerate(df_raw.columns, start=1):
    print(f"{index}. {column}")


# ------------------------------------------------------------
# 6. Data types
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DATA TYPES")
print("-" * 60)

print(df_raw.dtypes)


# ------------------------------------------------------------
# 7. Missing-value analysis
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("MISSING VALUE ANALYSIS")
print("-" * 60)

missing_counts = df_raw.isna().sum()

missing_table = pd.DataFrame({
    "column": missing_counts.index,
    "missing_count": missing_counts.values,
    "missing_percentage": (
        missing_counts.values / len(df_raw) * 100
    )
})

print(missing_table.to_string(index=False))


# ------------------------------------------------------------
# 8. Duplicate analysis
# ------------------------------------------------------------

try:
    duplicate_count = df_raw.duplicated().sum()

    print("\n" + "-" * 60)
    print("DUPLICATE ANALYSIS")
    print("-" * 60)

    print(f"Duplicate rows: {duplicate_count:,}")

except Exception as e:
    print("[ERROR] Duplicate analysis failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 9. Sample records
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("SAMPLE RECORDS")
print("-" * 60)

display(df_raw.head())


# ------------------------------------------------------------
# 10. Dataset memory usage
# ------------------------------------------------------------

memory_mb = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)

print("\n" + "-" * 60)
print("MEMORY USAGE")
print("-" * 60)

print(f"DataFrame memory usage: {memory_mb:.2f} MB")


# ------------------------------------------------------------
# 11. Create dataset inspection checkpoint
# ------------------------------------------------------------

dataset_inspection = {
    "dataset_path": str(DATASET_PATH),
    "dataset_name": CONFIG["project"]["dataset_name"],
    "dataset_version": CONFIG["project"]["dataset_version"],
    "rows": int(n_rows),
    "columns": int(n_columns),
    "column_names": list(df_raw.columns),
    "data_types": {
        column: str(dtype)
        for column, dtype in df_raw.dtypes.items()
    },
    "missing_values": {
        column: int(count)
        for column, count in missing_counts.items()
    },
    "duplicate_rows": int(duplicate_count),
    "memory_mb": round(memory_mb, 4),
}

DATASET_INSPECTION_CHECKPOINT = (
    DIRS["checkpoints"] / "dataset_inspection.json"
)

try:
    with open(
        DATASET_INSPECTION_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            dataset_inspection,
            f,
            indent=4
        )

    print("\n[SUCCESS] Dataset inspection checkpoint saved.")
    print(
        f"[INFO] Checkpoint: "
        f"{DATASET_INSPECTION_CHECKPOINT}"
    )

except Exception as e:
    print("[ERROR] Failed to save dataset inspection checkpoint.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 12. Final validation summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET VALIDATION SUMMARY")
print("=" * 60)

print(f"Records              : {n_rows:,}")
print(f"Columns              : {n_columns}")
print(f"Missing values       : {int(missing_counts.sum()):,}")
print(f"Duplicate rows       : {duplicate_count:,}")
print(f"Memory usage         : {memory_mb:.2f} MB")
print(f"Original CSV modified: NO")

print("=" * 60)
print("[SUCCESS] Dataset inspection completed.")
print("=" * 60)

DATASET LOADING & VALIDATION
[SUCCESS] Dataset found:
         E:\rag\synthetic_knowledge_items.csv

[SUCCESS] Dataset loaded successfully.

------------------------------------------------------------
DATASET DIMENSIONS
------------------------------------------------------------
Rows    : 100
Columns : 4
[SUCCESS] Record count matches expected value: 100
[SUCCESS] Column count matches expected value: 4

------------------------------------------------------------
COLUMN INFORMATION
------------------------------------------------------------
1. ki_topic
2. ki_text
3. alt_ki_text
4. bad_ki_text

------------------------------------------------------------
DATA TYPES
------------------------------------------------------------
ki_topic       str
ki_text        str
alt_ki_text    str
bad_ki_text    str
dtype: object

------------------------------------------------------------
MISSING VALUE ANALYSIS
------------------------------------------------------------
     column  missing_count 

,ki_topic,ki_text,alt_ki_text,bad_ki_text
0,Setting Up a Mobile Device for Company Email,**Setting Up a Mobile Device for Company Email...,"To set up a mobile device for company email, f...",# Setting Up a Mobile Device for Company Email...
1,Resetting a Forgotten PIN,**Resetting a Forgotten PIN**\n\nIf you have f...,"If you have forgotten your PIN, you can reset ...","# How to Resetting Your Forgot PIN \n\nSo, you..."
2,Configuring VPN Access for Remote Workers,**Configuring VPN Access for Remote Workers**\...,To configure VPN access for remote workers at ...,# How to Set Up VPN Access for Remote Workrs\n...
3,Troubleshooting Issues with Microsoft Office,**Troubleshooting Issues with Microsoft Office...,When troubleshooting issues with Microsoft Off...,# Troubleshooting Issues with Microsoft Office...
4,Setting Up a Conference Call on Cisco Webex,"To set up a conference call on Cisco Webex, fo...","To set up a conference call on Cisco Webex, fo...",# How To Set Up A Conference Call on Cisco Web...



------------------------------------------------------------
MEMORY USAGE
------------------------------------------------------------
DataFrame memory usage: 1.35 MB

[SUCCESS] Dataset inspection checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\dataset_inspection.json

DATASET VALIDATION SUMMARY
Records              : 100
Columns              : 4
Missing values       : 0
Duplicate rows       : 0
Memory usage         : 1.35 MB
Original CSV modified: NO
[SUCCESS] Dataset inspection completed.


In [31]:
# ============================================================
# CELL 4 — DATASET PROFILING & RAG FIELD DEFINITION
# ============================================================

import pandas as pd
import json
from pathlib import Path

print("=" * 60)
print("DATASET PROFILING & RAG FIELD DEFINITION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Define the dataset schema
# ------------------------------------------------------------

RAG_SCHEMA = {
    "document_id_source": "row_index",

    "topic_column": "ki_topic",

    "primary_text_column": "ki_text",

    "alternative_text_column": "alt_ki_text",

    "bad_text_column": "bad_ki_text",
}


# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

required_columns = [
    RAG_SCHEMA["topic_column"],
    RAG_SCHEMA["primary_text_column"],
    RAG_SCHEMA["alternative_text_column"],
    RAG_SCHEMA["bad_text_column"],
]

try:
    missing_columns = [
        column
        for column in required_columns
        if column not in df_raw.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required columns are missing: {missing_columns}"
        )

    print("[SUCCESS] All required dataset columns are present.")

except Exception as e:
    print("[ERROR] Dataset schema validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 3. Create a non-destructive profiling copy
# ------------------------------------------------------------

try:
    df_profile = df_raw.copy(deep=True)

    print("[SUCCESS] Profiling copy created.")
    print("[INFO] Original dataset remains unchanged.")

except Exception as e:
    print("[ERROR] Failed to create profiling copy.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Calculate character lengths
# ------------------------------------------------------------

try:
    df_profile["topic_char_length"] = (
        df_profile[RAG_SCHEMA["topic_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["primary_char_length"] = (
        df_profile[RAG_SCHEMA["primary_text_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["alternative_char_length"] = (
        df_profile[RAG_SCHEMA["alternative_text_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["bad_char_length"] = (
        df_profile[RAG_SCHEMA["bad_text_column"]]
        .astype(str)
        .str.len()
    )

    print("[SUCCESS] Text-length profiling completed.")

except Exception as e:
    print("[ERROR] Text-length calculation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 5. Generate descriptive statistics
# ------------------------------------------------------------

length_columns = [
    "topic_char_length",
    "primary_char_length",
    "alternative_char_length",
    "bad_char_length",
]

try:
    length_statistics = (
        df_profile[length_columns]
        .describe()
        .round(2)
    )

    print("\n" + "-" * 60)
    print("TEXT LENGTH STATISTICS")
    print("-" * 60)

    display(length_statistics)

except Exception as e:
    print("[ERROR] Failed to calculate text statistics.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 6. Display min/max examples
# ------------------------------------------------------------

try:
    print("\n" + "-" * 60)
    print("TEXT LENGTH RANGE")
    print("-" * 60)

    for column in length_columns:
        print(
            f"{column:25s}: "
            f"min={df_profile[column].min():,} | "
            f"max={df_profile[column].max():,} | "
            f"mean={df_profile[column].mean():,.2f}"
        )

except Exception as e:
    print("[ERROR] Failed to display length ranges.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 7. Check whether text fields are actually distinct
# ------------------------------------------------------------

try:
    primary_equals_alternative = (
        df_raw[RAG_SCHEMA["primary_text_column"]]
        ==
        df_raw[RAG_SCHEMA["alternative_text_column"]]
    ).sum()

    primary_equals_bad = (
        df_raw[RAG_SCHEMA["primary_text_column"]]
        ==
        df_raw[RAG_SCHEMA["bad_text_column"]]
    ).sum()

    alternative_equals_bad = (
        df_raw[RAG_SCHEMA["alternative_text_column"]]
        ==
        df_raw[RAG_SCHEMA["bad_text_column"]]
    ).sum()

    print("\n" + "-" * 60)
    print("TEXT VERSION COMPARISON")
    print("-" * 60)

    print(
        f"Primary == Alternative : "
        f"{primary_equals_alternative:,} / {len(df_raw):,}"
    )

    print(
        f"Primary == Bad         : "
        f"{primary_equals_bad:,} / {len(df_raw):,}"
    )

    print(
        f"Alternative == Bad     : "
        f"{alternative_equals_bad:,} / {len(df_raw):,}"
    )

except Exception as e:
    print("[ERROR] Text version comparison failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 8. Inspect topics
# ------------------------------------------------------------

try:
    unique_topics = df_raw[
        RAG_SCHEMA["topic_column"]
    ].nunique()

    print("\n" + "-" * 60)
    print("TOPIC ANALYSIS")
    print("-" * 60)

    print(f"Total records : {len(df_raw):,}")
    print(f"Unique topics : {unique_topics:,}")

    if unique_topics != len(df_raw):
        print(
            "[WARNING] Multiple records share the same topic."
        )
    else:
        print(
            "[SUCCESS] Every knowledge item has a unique topic."
        )

except Exception as e:
    print("[ERROR] Topic analysis failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 9. Define baseline RAG document strategy
# ------------------------------------------------------------

DOCUMENT_STRATEGY = {
    "baseline_corpus": "primary",
    "topic_field": RAG_SCHEMA["topic_column"],
    "content_field": RAG_SCHEMA["primary_text_column"],

    "excluded_from_baseline": [
        RAG_SCHEMA["alternative_text_column"],
        RAG_SCHEMA["bad_text_column"],
    ],

    "alternative_text_usage": (
        "Reserved for controlled experiments/evaluation."
    ),

    "bad_text_usage": (
        "Reserved for robustness/adversarial experiments "
        "and evaluation."
    ),
}


# ------------------------------------------------------------
# 10. Save schema and profiling checkpoint
# ------------------------------------------------------------

PROFILE_CHECKPOINT = (
    DIRS["checkpoints"] / "dataset_profile.json"
)

profile_checkpoint = {
    "schema": RAG_SCHEMA,
    "document_strategy": DOCUMENT_STRATEGY,

    "record_count": int(len(df_raw)),
    "unique_topics": int(unique_topics),

    "text_statistics": {
        column: {
            metric: float(length_statistics.loc[metric, column])
            for metric in length_statistics.index
        }
        for column in length_statistics.columns
    },

    "version_comparison": {
        "primary_equals_alternative": int(
            primary_equals_alternative
        ),
        "primary_equals_bad": int(
            primary_equals_bad
        ),
        "alternative_equals_bad": int(
            alternative_equals_bad
        ),
    },
}

try:
    with open(
        PROFILE_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            profile_checkpoint,
            f,
            indent=4
        )

    print("\n[SUCCESS] Dataset profile checkpoint saved.")
    print(f"[INFO] Checkpoint: {PROFILE_CHECKPOINT}")

except Exception as e:
    print("[ERROR] Failed to save dataset profile checkpoint.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 11. Final strategy summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BASELINE RAG DOCUMENT STRATEGY")
print("=" * 60)

print(f"Topic field       : {RAG_SCHEMA['topic_column']}")
print(f"Primary text      : {RAG_SCHEMA['primary_text_column']}")
print(f"Alternative text  : {RAG_SCHEMA['alternative_text_column']}")
print(f"Bad text          : {RAG_SCHEMA['bad_text_column']}")

print("\nBaseline corpus:")
print("  ki_topic + ki_text")

print("\nExcluded from baseline:")
print("  alt_ki_text")
print("  bad_ki_text")

print("\n[SUCCESS] Dataset profiling completed.")
print("=" * 60)

DATASET PROFILING & RAG FIELD DEFINITION
[SUCCESS] All required dataset columns are present.
[SUCCESS] Profiling copy created.
[INFO] Original dataset remains unchanged.
[SUCCESS] Text-length profiling completed.

------------------------------------------------------------
TEXT LENGTH STATISTICS
------------------------------------------------------------


,topic_char_length,primary_char_length,alternative_char_length,bad_char_length
count,100.00,100.00,100.00,100.00
mean,41.55,2581.34,2440.64,2921.60
std,9.77,361.60,372.04,319.16
min,25.00,1606.00,1526.00,2326.00
25%,33.00,2327.25,2240.50,2756.25
50%,41.00,2573.50,2415.00,2923.00
75%,50.00,2824.25,2660.50,3108.00
max,64.00,3730.00,3456.00,3904.00



------------------------------------------------------------
TEXT LENGTH RANGE
------------------------------------------------------------
topic_char_length        : min=25 | max=64 | mean=41.55
primary_char_length      : min=1,606 | max=3,730 | mean=2,581.34
alternative_char_length  : min=1,526 | max=3,456 | mean=2,440.64
bad_char_length          : min=2,326 | max=3,904 | mean=2,921.60

------------------------------------------------------------
TEXT VERSION COMPARISON
------------------------------------------------------------
Primary == Alternative : 0 / 100
Primary == Bad         : 0 / 100
Alternative == Bad     : 0 / 100

------------------------------------------------------------
TOPIC ANALYSIS
------------------------------------------------------------
Total records : 100
Unique topics : 98
[WARNING] Multiple records share the same topic.

[SUCCESS] Dataset profile checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\dataset_profile.json

BASELINE RAG DOCUMENT STRATEGY


In [32]:
# ============================================================
# CELL 5 — RAG DOCUMENT CREATION & PERSISTENCE
# ============================================================

import json
import re
import gc
from pathlib import Path

print("=" * 60)
print("RAG DOCUMENT CREATION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TOPIC_COLUMN = RAG_SCHEMA["topic_column"]
TEXT_COLUMN = RAG_SCHEMA["primary_text_column"]

DOCUMENTS_CHECKPOINT = (
    DIRS["data_processed"] / "rag_documents.json"
)


# ------------------------------------------------------------
# 2. Text cleaning function
# ------------------------------------------------------------

def clean_document_text(text):
    """
    Perform conservative text cleaning.

    The purpose is to remove accidental formatting noise
    while preserving the actual knowledge content.
    """

    if text is None:
        return ""

    text = str(text)

    # Normalize Windows line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove trailing whitespace from each line
    text = "\n".join(
        line.rstrip()
        for line in text.split("\n")
    )

    # Collapse excessive blank lines
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    # Remove unnecessary leading/trailing whitespace
    text = text.strip()

    return text


# ------------------------------------------------------------
# 3. Validate source data before document creation
# ------------------------------------------------------------

try:
    if df_raw.empty:
        raise ValueError(
            "Source DataFrame is empty."
        )

    if TOPIC_COLUMN not in df_raw.columns:
        raise KeyError(
            f"Topic column '{TOPIC_COLUMN}' not found."
        )

    if TEXT_COLUMN not in df_raw.columns:
        raise KeyError(
            f"Text column '{TEXT_COLUMN}' not found."
        )

    if df_raw[TOPIC_COLUMN].isna().any():
        raise ValueError(
            "Topic column contains missing values."
        )

    if df_raw[TEXT_COLUMN].isna().any():
        raise ValueError(
            "Primary text column contains missing values."
        )

    print("[SUCCESS] Source dataset validation passed.")

except Exception as e:
    print("[ERROR] Source validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Create RAG documents
# ------------------------------------------------------------

rag_documents = []

try:

    for row_index, row in df_raw.iterrows():

        topic = str(row[TOPIC_COLUMN]).strip()
        original_text = str(row[TEXT_COLUMN])

        cleaned_text = clean_document_text(
            original_text
        )

        # Validate cleaned content
        if not topic:
            raise ValueError(
                f"Empty topic at row {row_index}."
            )

        if not cleaned_text:
            raise ValueError(
                f"Empty document text at row {row_index}."
            )

        document_id = f"doc_{row_index + 1:03d}"

        document = {
            "document_id": document_id,

            "metadata": {
                "topic": topic,
                "source": CONFIG["project"]["dataset_name"],
                "dataset_version": (
                    CONFIG["project"]["dataset_version"]
                ),
                "original_row_index": int(row_index),
            },

            "text": cleaned_text,
        }

        rag_documents.append(document)

    print(
        f"[SUCCESS] Created {len(rag_documents):,} "
        f"RAG documents."
    )

except Exception as e:
    print("[ERROR] RAG document creation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 5. Validate generated documents
# ------------------------------------------------------------

try:

    if len(rag_documents) != len(df_raw):
        raise ValueError(
            "Document count does not match source "
            "record count."
        )

    document_ids = [
        document["document_id"]
        for document in rag_documents
    ]

    if len(document_ids) != len(set(document_ids)):
        raise ValueError(
            "Duplicate document IDs detected."
        )

    for document in rag_documents:

        required_keys = {
            "document_id",
            "metadata",
            "text",
        }

        if not required_keys.issubset(
            document.keys()
        ):
            raise ValueError(
                f"Invalid document structure: "
                f"{document.get('document_id')}"
            )

        if not document["text"].strip():
            raise ValueError(
                f"Empty text in "
                f"{document['document_id']}"
            )

        if not document["metadata"]["topic"].strip():
            raise ValueError(
                f"Empty topic in "
                f"{document['document_id']}"
            )

    print("[SUCCESS] Document validation passed.")
    print(f"[INFO] Valid documents: {len(rag_documents):,}")

except Exception as e:
    print("[ERROR] Document validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 6. Calculate document statistics
# ------------------------------------------------------------

try:

    document_lengths = [
        len(document["text"])
        for document in rag_documents
    ]

    total_characters = sum(
        document_lengths
    )

    average_length = (
        total_characters / len(document_lengths)
    )

    min_length = min(document_lengths)
    max_length = max(document_lengths)

    print("\n" + "-" * 60)
    print("PREPARED DOCUMENT STATISTICS")
    print("-" * 60)

    print(
        f"Documents           : "
        f"{len(rag_documents):,}"
    )

    print(
        f"Total characters    : "
        f"{total_characters:,}"
    )

    print(
        f"Average characters  : "
        f"{average_length:,.2f}"
    )

    print(
        f"Minimum characters  : "
        f"{min_length:,}"
    )

    print(
        f"Maximum characters  : "
        f"{max_length:,}"
    )

except Exception as e:
    print("[ERROR] Document statistics failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 7. Preview prepared documents
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DOCUMENT PREVIEW")
print("-" * 60)

for document in rag_documents[:3]:

    print(
        f"\nDocument ID : "
        f"{document['document_id']}"
    )

    print(
        f"Topic       : "
        f"{document['metadata']['topic']}"
    )

    preview = document["text"][:500]

    print("Text preview:")
    print(preview)

    if len(document["text"]) > 500:
        print("...")


# ------------------------------------------------------------
# 8. Save documents using temporary file
# ------------------------------------------------------------

TEMP_DOCUMENTS_CHECKPOINT = (
    DOCUMENTS_CHECKPOINT.with_suffix(".tmp")
)

try:

    with open(
        TEMP_DOCUMENTS_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            rag_documents,
            f,
            ensure_ascii=False,
            indent=2
        )

    # Replace old checkpoint only after successful write
    TEMP_DOCUMENTS_CHECKPOINT.replace(
        DOCUMENTS_CHECKPOINT
    )

    print(
        "\n[SUCCESS] RAG documents checkpoint saved."
    )

    print(
        f"[INFO] Path: {DOCUMENTS_CHECKPOINT}"
    )

except Exception as e:

    # Remove incomplete temporary file
    if TEMP_DOCUMENTS_CHECKPOINT.exists():
        TEMP_DOCUMENTS_CHECKPOINT.unlink(
            missing_ok=True
        )

    print(
        "[ERROR] Failed to save RAG document checkpoint."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 9. Final integrity summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RAG DOCUMENT PREPARATION COMPLETE")
print("=" * 60)

print(
    f"Source records       : {len(df_raw):,}"
)

print(
    f"Prepared documents   : {len(rag_documents):,}"
)

print(
    f"Unique document IDs  : "
    f"{len(set(document_ids)):,}"
)

print(
    f"Baseline text field  : {TEXT_COLUMN}"
)

print(
    "Alternative text     : NOT included"
)

print(
    "Bad text             : NOT included"
)

print(
    "Original dataset     : UNCHANGED"
)

print("=" * 60)
print("[SUCCESS] Document layer is ready.")
print("=" * 60)


# ------------------------------------------------------------
# 10. Memory cleanup
# ------------------------------------------------------------

del df_profile
gc.collect()

print("[INFO] Temporary profiling objects released.")

RAG DOCUMENT CREATION
[SUCCESS] Source dataset validation passed.
[SUCCESS] Created 100 RAG documents.
[SUCCESS] Document validation passed.
[INFO] Valid documents: 100

------------------------------------------------------------
PREPARED DOCUMENT STATISTICS
------------------------------------------------------------
Documents           : 100
Total characters    : 258,134
Average characters  : 2,581.34
Minimum characters  : 1,606
Maximum characters  : 3,730

------------------------------------------------------------
DOCUMENT PREVIEW
------------------------------------------------------------

Document ID : doc_001
Topic       : Setting Up a Mobile Device for Company Email
Text preview:
**Setting Up a Mobile Device for Company Email**

**Prerequisites:**

* Mobile device with a supported operating system (iOS, Android, or Windows)
* Company email account credentials
* Mobile device management (MDM) profile installed (if required by company policy)

**Step 1: Ensure Mobile Device Ma

In [33]:
# ============================================================
# CELL 6 — TOKENIZATION & CHUNKING FEASIBILITY ANALYSIS
# ============================================================

import json
import gc
from pathlib import Path

print("=" * 60)
print("TOKENIZATION & CHUNKING FEASIBILITY ANALYSIS")
print("=" * 60)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DOCUMENTS_CHECKPOINT = (
    DIRS["data_processed"] / "rag_documents.json"
)

CHUNK_SIZE = CONFIG["chunking"]["chunk_size"]
CHUNK_OVERLAP = CONFIG["chunking"]["chunk_overlap"]

TOKEN_ANALYSIS_CHECKPOINT = (
    DIRS["checkpoints"] / "tokenization_analysis.json"
)


# ------------------------------------------------------------
# 2. Load persisted documents
# ------------------------------------------------------------

try:

    if not DOCUMENTS_CHECKPOINT.exists():
        raise FileNotFoundError(
            f"Document checkpoint not found: "
            f"{DOCUMENTS_CHECKPOINT}"
        )

    with open(
        DOCUMENTS_CHECKPOINT,
        "r",
        encoding="utf-8"
    ) as f:

        persisted_documents = json.load(f)

    if not persisted_documents:
        raise ValueError(
            "Persisted document list is empty."
        )

    print(
        f"[SUCCESS] Loaded {len(persisted_documents):,} "
        "persisted RAG documents."
    )

except Exception as e:

    print(
        "[ERROR] Failed to load persisted RAG documents."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 3. Load tokenizer
# ------------------------------------------------------------

tokenizer = None
TOKENIZER_NAME = None

try:

    # tiktoken provides a stable tokenizer for
    # token-count analysis.
    import tiktoken

    try:
        tokenizer = tiktoken.get_encoding("cl100k_base")
        TOKENIZER_NAME = "cl100k_base"

    except Exception:

        tokenizer = tiktoken.get_encoding(
            "o200k_base"
        )
        TOKENIZER_NAME = "o200k_base"

    print(
        f"[SUCCESS] Tokenizer loaded: "
        f"{TOKENIZER_NAME}"
    )

except ImportError:

    print(
        "[WARNING] tiktoken is not installed."
    )

    print(
        "[INFO] Installing/loading a tokenizer will "
        "be handled before chunk generation."
    )

except Exception as e:

    print(
        "[WARNING] Tokenizer initialization failed."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )


# ------------------------------------------------------------
# 4. Token count analysis
# ------------------------------------------------------------

try:

    if tokenizer is None:

        raise RuntimeError(
            "No tokenizer is available. "
            "Cannot perform token-level analysis."
        )

    token_statistics = []

    for document in persisted_documents:

        text = document["text"]

        token_ids = tokenizer.encode(
            text,
            disallowed_special=()
        )

        token_count = len(token_ids)

        character_count = len(text)

        token_statistics.append({
            "document_id": document["document_id"],
            "topic": document["metadata"]["topic"],
            "character_count": character_count,
            "token_count": token_count,
            "characters_per_token": (
                character_count / token_count
                if token_count > 0
                else 0
            ),
        })

    print(
        "[SUCCESS] Token counts calculated for all "
        f"{len(token_statistics):,} documents."
    )

except Exception as e:

    print(
        "[ERROR] Token analysis failed."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 5. Statistical analysis
# ------------------------------------------------------------

try:

    token_counts = [
        item["token_count"]
        for item in token_statistics
    ]

    character_counts = [
        item["character_count"]
        for item in token_statistics
    ]

    ratios = [
        item["characters_per_token"]
        for item in token_statistics
    ]

    token_counts_sorted = sorted(token_counts)

    def percentile(values, percentage):

        if not values:
            return 0

        position = (
            (len(values) - 1)
            * percentage
        )

        lower = int(position)
        upper = min(
            lower + 1,
            len(values) - 1
        )

        fraction = position - lower

        return (
            values[lower]
            + (
                values[upper]
                - values[lower]
            ) * fraction
        )

    token_analysis = {
        "document_count": len(token_counts),

        "tokenizer": TOKENIZER_NAME,

        "chunk_size": CHUNK_SIZE,

        "chunk_overlap": CHUNK_OVERLAP,

        "token_statistics": {
            "minimum": min(token_counts),
            "maximum": max(token_counts),
            "mean": sum(token_counts)
            / len(token_counts),
            "median": percentile(
                token_counts_sorted,
                0.50
            ),
            "p25": percentile(
                token_counts_sorted,
                0.25
            ),
            "p75": percentile(
                token_counts_sorted,
                0.75
            ),
        },

        "character_statistics": {
            "minimum": min(character_counts),
            "maximum": max(character_counts),
            "mean": sum(character_counts)
            / len(character_counts),
        },

        "characters_per_token": {
            "minimum": min(ratios),
            "maximum": max(ratios),
            "mean": sum(ratios)
            / len(ratios),
        },
    }

except Exception as e:

    print(
        "[ERROR] Failed to calculate token statistics."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 6. Estimate chunk counts
# ------------------------------------------------------------

try:

    estimated_chunks = []

    for item in token_statistics:

        token_count = item["token_count"]

        if token_count <= CHUNK_SIZE:
            chunks_needed = 1

        else:

            effective_step = (
                CHUNK_SIZE - CHUNK_OVERLAP
            )

            chunks_needed = (
                1
                + (
                    token_count
                    - CHUNK_SIZE
                    + effective_step
                    - 1
                )
                // effective_step
            )

        estimated_chunks.append({
            "document_id": item["document_id"],
            "token_count": token_count,
            "estimated_chunks": int(
                chunks_needed
            ),
        })

    total_estimated_chunks = sum(
        item["estimated_chunks"]
        for item in estimated_chunks
    )

    token_analysis[
        "estimated_chunk_count"
    ] = total_estimated_chunks

except Exception as e:

    print(
        "[ERROR] Chunk estimation failed."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("TOKENIZATION RESULTS")
print("-" * 60)

print(
    f"Tokenizer             : "
    f"{TOKENIZER_NAME}"
)

print(
    f"Documents             : "
    f"{len(token_counts):,}"
)

print(
    f"Minimum tokens        : "
    f"{min(token_counts):,}"
)

print(
    f"Maximum tokens        : "
    f"{max(token_counts):,}"
)

print(
    f"Average tokens        : "
    f"{sum(token_counts) / len(token_counts):,.2f}"
)

print(
    f"Median tokens         : "
    f"{token_analysis['token_statistics']['median']:,.0f}"
)

print(
    f"Chunk size            : "
    f"{CHUNK_SIZE} tokens"
)

print(
    f"Chunk overlap         : "
    f"{CHUNK_OVERLAP} tokens"
)

print(
    f"Estimated total chunks: "
    f"{total_estimated_chunks:,}"
)


# ------------------------------------------------------------
# 8. Document-level token distribution
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DOCUMENT TOKEN DISTRIBUTION")
print("-" * 60)

for item in token_statistics[:10]:

    print(
        f"{item['document_id']} | "
        f"{item['token_count']:,} tokens | "
        f"{item['character_count']:,} chars | "
        f"{item['characters_per_token']:.2f} chars/token"
    )


# ------------------------------------------------------------
# 9. Save analysis checkpoint
# ------------------------------------------------------------

token_analysis[
    "document_token_statistics"
] = token_statistics

token_analysis[
    "estimated_document_chunks"
] = estimated_chunks

TEMP_TOKEN_CHECKPOINT = (
    TOKEN_ANALYSIS_CHECKPOINT.with_suffix(
        ".tmp"
    )
)

try:

    with open(
        TEMP_TOKEN_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            token_analysis,
            f,
            indent=4
        )

    TEMP_TOKEN_CHECKPOINT.replace(
        TOKEN_ANALYSIS_CHECKPOINT
    )

    print(
        "\n[SUCCESS] Tokenization analysis "
        "checkpoint saved."
    )

    print(
        f"[INFO] Path: "
        f"{TOKEN_ANALYSIS_CHECKPOINT}"
    )

except Exception as e:

    if TEMP_TOKEN_CHECKPOINT.exists():
        TEMP_TOKEN_CHECKPOINT.unlink(
            missing_ok=True
        )

    print(
        "[ERROR] Failed to save tokenization "
        "checkpoint."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOKENIZATION ANALYSIS COMPLETE")
print("=" * 60)

print(
    "[SUCCESS] No documents were modified."
)

print(
    "[SUCCESS] No chunks were generated yet."
)

print(
    "[SUCCESS] Tokenization analysis persisted."
)

print("=" * 60)


# ------------------------------------------------------------
# 11. Memory cleanup
# ------------------------------------------------------------

del token_ids
gc.collect()

print("[INFO] Temporary tokenization objects released.")

TOKENIZATION & CHUNKING FEASIBILITY ANALYSIS
[SUCCESS] Loaded 100 persisted RAG documents.
[SUCCESS] Tokenizer loaded: cl100k_base
[SUCCESS] Token counts calculated for all 100 documents.

------------------------------------------------------------
TOKENIZATION RESULTS
------------------------------------------------------------
Tokenizer             : cl100k_base
Documents             : 100
Minimum tokens        : 335
Maximum tokens        : 776
Average tokens        : 537.08
Median tokens         : 537
Chunk size            : 500 tokens
Chunk overlap         : 50 tokens
Estimated total chunks: 164

------------------------------------------------------------
DOCUMENT TOKEN DISTRIBUTION
------------------------------------------------------------
doc_001 | 535 tokens | 2,568 chars | 4.80 chars/token
doc_002 | 421 tokens | 1,873 chars | 4.45 chars/token
doc_003 | 528 tokens | 2,607 chars | 4.94 chars/token
doc_004 | 558 tokens | 2,924 chars | 5.24 chars/token
doc_005 | 523 tokens | 2,

In [34]:
# ============================================================
# CELL 7 — CORRECTED TOKEN-AWARE CHUNK GENERATION
# ============================================================

import os
import json
import gc
import tempfile
from pathlib import Path
from statistics import mean, median

print("=" * 60)
print("CORRECTED TOKEN-AWARE CHUNK GENERATION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

DOCUMENTS_PATH = (
    PROJECT_ROOT / "data" / "processed" / "rag_documents.json"
)

CHUNKS_DIR = PROJECT_ROOT / "data" / "processed"
CHUNKS_PATH = CHUNKS_DIR / "rag_chunks.json"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHUNK_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "chunk_generation.json"
)

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

TOKENIZER_NAME = "cl100k_base"

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    output_path = Path(output_path)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(tmp_file.name)

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):
            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# STEP 1 — LOAD PERSISTED DOCUMENTS
# ------------------------------------------------------------

try:

    if not DOCUMENTS_PATH.exists():

        raise FileNotFoundError(
            f"Persisted document file not found:\n"
            f"{DOCUMENTS_PATH}"
        )

    with open(
        DOCUMENTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        documents = json.load(f)

    if not isinstance(documents, list):

        raise ValueError(
            "Persisted documents must be a list."
        )

    if len(documents) != 100:

        raise ValueError(
            f"Expected 100 documents, "
            f"found {len(documents)}."
        )

    print(
        f"[SUCCESS] Loaded {len(documents)} "
        f"persisted RAG documents."
    )

    # --------------------------------------------------------
    # STEP 2 — LOAD TOKENIZER
    # --------------------------------------------------------

    try:

        import tiktoken

    except ImportError as e:

        raise ImportError(
            "tiktoken is required.\n"
            "Install with: pip install tiktoken"
        ) from e

    tokenizer = tiktoken.get_encoding(
        TOKENIZER_NAME
    )

    print(
        f"[SUCCESS] Tokenizer loaded: "
        f"{TOKENIZER_NAME}"
    )

    # --------------------------------------------------------
    # STEP 3 — VALIDATE CONFIGURATION
    # --------------------------------------------------------

    if CHUNK_SIZE <= 0:

        raise ValueError(
            "CHUNK_SIZE must be greater than zero."
        )

    if CHUNK_OVERLAP < 0:

        raise ValueError(
            "CHUNK_OVERLAP cannot be negative."
        )

    if CHUNK_OVERLAP >= CHUNK_SIZE:

        raise ValueError(
            "CHUNK_OVERLAP must be smaller "
            "than CHUNK_SIZE."
        )

    STEP_SIZE = CHUNK_SIZE - CHUNK_OVERLAP

    print()
    print("-" * 60)
    print("CHUNKING CONFIGURATION")
    print("-" * 60)
    print(
        f"Tokenizer       : {TOKENIZER_NAME}"
    )
    print(
        f"Chunk size      : {CHUNK_SIZE} tokens"
    )
    print(
        f"Chunk overlap   : {CHUNK_OVERLAP} tokens"
    )
    print(
        f"Step size       : {STEP_SIZE} tokens"
    )

    # --------------------------------------------------------
    # STEP 4 — GENERATE CORRECTED CHUNKS
    # --------------------------------------------------------

    chunks = []

    for doc in documents:

        document_id = doc.get(
            "document_id"
        )

        document_text = doc.get(
            "text",
            ""
        )

        document_metadata = doc.get(
            "metadata",
            {}
        )

        if not document_id:

            raise ValueError(
                "Document missing document_id."
            )

        if not isinstance(
            document_text,
            str
        ):

            raise ValueError(
                f"{document_id}: text is not a string."
            )

        if not document_text.strip():

            raise ValueError(
                f"{document_id}: empty document text."
            )

        # Encode complete document.
        token_ids = tokenizer.encode(
            document_text
        )

        total_tokens = len(token_ids)

        if total_tokens == 0:

            raise ValueError(
                f"{document_id}: zero tokens."
            )

        chunk_number = 0
        start_token = 0

        while start_token < total_tokens:

            end_token = min(
                start_token + CHUNK_SIZE,
                total_tokens
            )

            chunk_token_ids = token_ids[
                start_token:end_token
            ]

            chunk_text = tokenizer.decode(
                chunk_token_ids
            ).strip()

            if not chunk_text:

                raise ValueError(
                    f"{document_id}: "
                    f"generated empty chunk."
                )

            chunk_number += 1

            chunk_id = (
                f"{document_id}"
                f"_chunk_{chunk_number:03d}"
            )

            chunks.append({
                "chunk_id": chunk_id,

                "document_id": document_id,

                "text": chunk_text,

                "metadata": {
                    **document_metadata,

                    "chunk_number": chunk_number,

                    "start_token": start_token,

                    "end_token": end_token,

                    "token_count": len(
                        chunk_token_ids
                    ),

                    "chunk_size": CHUNK_SIZE,

                    "chunk_overlap": CHUNK_OVERLAP,

                    "tokenizer": TOKENIZER_NAME
                }
            })

            # ------------------------------------------------
            # CRITICAL FIX
            #
            # If this chunk reached the end of the document,
            # stop immediately.
            #
            # This prevents tiny trailing fragments such as
            # 3 tokens from being created.
            # ------------------------------------------------

            if end_token >= total_tokens:

                break

            # Move forward while preserving overlap.
            start_token += STEP_SIZE

    # --------------------------------------------------------
    # STEP 5 — VALIDATE GENERATED CHUNKS
    # --------------------------------------------------------

    if not chunks:

        raise ValueError(
            "No chunks were generated."
        )

    chunk_ids = [
        chunk["chunk_id"]
        for chunk in chunks
    ]

    if len(chunk_ids) != len(
        set(chunk_ids)
    ):

        raise ValueError(
            "Duplicate chunk IDs detected."
        )

    source_document_ids = {
        doc["document_id"]
        for doc in documents
    }

    chunk_document_ids = {
        chunk["document_id"]
        for chunk in chunks
    }

    if not chunk_document_ids.issubset(
        source_document_ids
    ):

        raise ValueError(
            "Chunks reference unknown documents."
        )

    for chunk in chunks:

        token_count = chunk[
            "metadata"
        ]["token_count"]

        if token_count <= 0:

            raise ValueError(
                f"{chunk['chunk_id']}: "
                f"invalid token count."
            )

        if token_count > CHUNK_SIZE:

            raise ValueError(
                f"{chunk['chunk_id']}: "
                f"exceeds chunk size."
            )

        if not chunk["text"].strip():

            raise ValueError(
                f"{chunk['chunk_id']}: "
                f"empty text."
            )

    # --------------------------------------------------------
    # STEP 6 — CHECK FOR UNNECESSARY TINY FINAL CHUNKS
    # --------------------------------------------------------

    tiny_chunks = [
        chunk
        for chunk in chunks
        if chunk["metadata"]["token_count"] < 50
    ]

    # A tiny chunk is acceptable if the entire document itself
    # is tiny. Our documents are much larger, so a tiny chunk
    # indicates a generation problem.

    if tiny_chunks:

        raise ValueError(
            f"Found {len(tiny_chunks)} "
            f"unexpected tiny chunks (<50 tokens)."
        )

    print(
        "[SUCCESS] No unnecessary tiny trailing "
        "chunks detected."
    )

    # --------------------------------------------------------
    # STEP 7 — DOCUMENT CHUNK DISTRIBUTION
    # --------------------------------------------------------

    chunks_per_document = {}

    for chunk in chunks:

        doc_id = chunk["document_id"]

        chunks_per_document[doc_id] = (
            chunks_per_document.get(
                doc_id,
                0
            ) + 1
        )

    one_chunk_documents = sum(
        1
        for count in chunks_per_document.values()
        if count == 1
    )

    two_chunk_documents = sum(
        1
        for count in chunks_per_document.values()
        if count == 2
    )

    three_plus_documents = sum(
        1
        for count in chunks_per_document.values()
        if count >= 3
    )

    # --------------------------------------------------------
    # STEP 8 — TOKEN STATISTICS
    # --------------------------------------------------------

    chunk_token_counts = [
        chunk["metadata"]["token_count"]
        for chunk in chunks
    ]

    # --------------------------------------------------------
    # STEP 9 — ATOMICALLY SAVE CHUNKS
    # --------------------------------------------------------

    atomic_json_save(
        chunks,
        CHUNKS_PATH
    )

    print()
    print(
        "[SUCCESS] Corrected chunk file saved atomically."
    )

    print(
        f"[INFO] Path: {CHUNKS_PATH}"
    )

    # --------------------------------------------------------
    # STEP 10 — SAVE CHECKPOINT
    # --------------------------------------------------------

    checkpoint = {

        "stage":
            "token_aware_chunk_generation",

        "status":
            "success",

        "generation_version":
            "corrected_final_chunk_handling",

        "source_documents":
            len(documents),

        "generated_chunks":
            len(chunks),

        "tokenizer":
            TOKENIZER_NAME,

        "chunk_size_tokens":
            CHUNK_SIZE,

        "chunk_overlap_tokens":
            CHUNK_OVERLAP,

        "step_size_tokens":
            STEP_SIZE,

        "chunk_token_statistics": {

            "minimum":
                min(chunk_token_counts),

            "maximum":
                max(chunk_token_counts),

            "mean":
                mean(chunk_token_counts),

            "median":
                median(chunk_token_counts)
        },

        "document_distribution": {

            "one_chunk_documents":
                one_chunk_documents,

            "two_chunk_documents":
                two_chunk_documents,

            "three_plus_chunk_documents":
                three_plus_documents,

            "total_documents":
                len(documents)
        },

        "tiny_chunks":
            len(tiny_chunks),

        "source_documents_path":
            str(DOCUMENTS_PATH),

        "chunks_path":
            str(CHUNKS_PATH)
    }

    atomic_json_save(
        checkpoint,
        CHUNK_CHECKPOINT_PATH
    )

    # --------------------------------------------------------
    # STEP 11 — DISPLAY RESULTS
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("CORRECTED CHUNK GENERATION RESULTS")
    print("=" * 60)

    print(
        f"Source documents       : {len(documents)}"
    )

    print(
        f"Generated chunks       : {len(chunks)}"
    )

    print(
        f"Tokenizer              : {TOKENIZER_NAME}"
    )

    print(
        f"Chunk size             : "
        f"{CHUNK_SIZE} tokens"
    )

    print(
        f"Chunk overlap          : "
        f"{CHUNK_OVERLAP} tokens"
    )

    print(
        f"Step size              : "
        f"{STEP_SIZE} tokens"
    )

    print()
    print(
        f"Minimum chunk tokens   : "
        f"{min(chunk_token_counts)}"
    )

    print(
        f"Maximum chunk tokens   : "
        f"{max(chunk_token_counts)}"
    )

    print(
        f"Average chunk tokens   : "
        f"{mean(chunk_token_counts):.2f}"
    )

    print(
        f"Median chunk tokens    : "
        f"{median(chunk_token_counts):.0f}"
    )

    print()
    print(
        f"1-chunk documents      : "
        f"{one_chunk_documents}"
    )

    print(
        f"2-chunk documents      : "
        f"{two_chunk_documents}"
    )

    print(
        f"3+ chunk documents     : "
        f"{three_plus_documents}"
    )

    print()
    print("-" * 60)
    print("FIRST 10 CORRECTED CHUNKS")
    print("-" * 60)

    for chunk in chunks[:10]:

        preview = (
            chunk["text"][:100]
            .replace("\n", " ")
        )

        print(
            f"{chunk['chunk_id']} | "
            f"{chunk['metadata']['token_count']} "
            f"tokens | {preview}..."
        )

    print()
    print(
        "[SUCCESS] Chunk generation checkpoint saved."
    )

    print(
        f"[INFO] Checkpoint: "
        f"{CHUNK_CHECKPOINT_PATH}"
    )

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    del documents
    del chunks
    del tokenizer
    del token_ids
    del chunk_token_counts
    del chunks_per_document
    del tiny_chunks

    gc.collect()

    print()
    print(
        "[INFO] Temporary chunk-generation "
        "objects released."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] CORRECTED CHUNK GENERATION FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] Source documents were not modified."
    )

    print(
        "[INFO] Existing chunks were not accepted "
        "as a successful generation."
    )

    raise

CORRECTED TOKEN-AWARE CHUNK GENERATION


[SUCCESS] Loaded 100 persisted RAG documents.
[SUCCESS] Tokenizer loaded: cl100k_base

------------------------------------------------------------
CHUNKING CONFIGURATION
------------------------------------------------------------
Tokenizer       : cl100k_base
Chunk size      : 500 tokens
Chunk overlap   : 50 tokens
Step size       : 450 tokens
[SUCCESS] No unnecessary tiny trailing chunks detected.

[SUCCESS] Corrected chunk file saved atomically.
[INFO] Path: E:\rag\data\processed\rag_chunks.json

CORRECTED CHUNK GENERATION RESULTS
Source documents       : 100
Generated chunks       : 164
Tokenizer              : cl100k_base
Chunk size             : 500 tokens
Chunk overlap          : 50 tokens
Step size              : 450 tokens

Minimum chunk tokens   : 53
Maximum chunk tokens   : 500
Average chunk tokens   : 347.00
Median chunk tokens    : 474

1-chunk documents      : 36
2-chunk documents      : 64
3+ chunk documents     : 0

-----------------------------------------------------

In [35]:
# ============================================================
# CELL 8 — CHUNK QUALITY & INTEGRITY ANALYSIS
# ============================================================

import os
import json
import gc
import tempfile
from pathlib import Path
from collections import Counter, defaultdict
from statistics import mean, median

print("=" * 60)
print("CHUNK QUALITY & INTEGRITY ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "rag_chunks.json"
ANALYSIS_PATH = PROJECT_ROOT / "checkpoints" / "chunk_quality_analysis.json"

EXPECTED_DOCUMENTS = 100
EXPECTED_CHUNKS = 189

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(tmp_file.name)

        os.replace(temp_path, output_path)

    except Exception:
        if temp_path is not None and temp_path.exists():
            try:
                temp_path.unlink()
            except Exception:
                pass
        raise


# ------------------------------------------------------------
# STEP 1 — LOAD CHUNKS
# ------------------------------------------------------------

try:

    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(
            f"Chunk file not found: {CHUNKS_PATH}"
        )

    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    if not isinstance(chunks, list):
        raise ValueError("Chunk file must contain a list.")

    print(f"[SUCCESS] Loaded {len(chunks)} persisted chunks.")

    # --------------------------------------------------------
    # STEP 2 — BASIC STRUCTURAL VALIDATION
    # --------------------------------------------------------

    required_keys = {
        "chunk_id",
        "document_id",
        "text",
        "metadata"
    }

    chunk_ids = []
    document_ids = []

    for chunk in chunks:

        if not isinstance(chunk, dict):
            raise ValueError("Invalid chunk object detected.")

        if not required_keys.issubset(chunk.keys()):
            raise ValueError(
                f"Missing required keys in chunk: "
                f"{chunk.get('chunk_id', 'UNKNOWN')}"
            )

        if not isinstance(chunk["text"], str):
            raise ValueError(
                f"Non-string text in {chunk['chunk_id']}"
            )

        if not chunk["text"].strip():
            raise ValueError(
                f"Empty text in {chunk['chunk_id']}"
            )

        if not chunk["document_id"]:
            raise ValueError(
                f"Missing document_id in {chunk['chunk_id']}"
            )

        chunk_ids.append(chunk["chunk_id"])
        document_ids.append(chunk["document_id"])

    # Unique chunk IDs
    if len(chunk_ids) != len(set(chunk_ids)):
        raise ValueError("Duplicate chunk IDs detected.")

    print("[SUCCESS] Structural validation passed.")
    print("[SUCCESS] Chunk IDs are unique.")

    # --------------------------------------------------------
    # STEP 3 — TOKEN STATISTICS
    # --------------------------------------------------------

    token_counts = []
    character_counts = []

    for chunk in chunks:

        metadata = chunk["metadata"]

        token_count = metadata.get("token_count")

        if not isinstance(token_count, int):
            raise ValueError(
                f"Invalid token_count in {chunk['chunk_id']}"
            )

        if token_count <= 0:
            raise ValueError(
                f"Non-positive token_count in {chunk['chunk_id']}"
            )

        if token_count > CHUNK_SIZE:
            raise ValueError(
                f"{chunk['chunk_id']} exceeds "
                f"{CHUNK_SIZE} token limit."
            )

        token_counts.append(token_count)
        character_counts.append(len(chunk["text"]))

    # --------------------------------------------------------
    # STEP 4 — DOCUMENT → CHUNK DISTRIBUTION
    # --------------------------------------------------------

    chunks_per_document = Counter(document_ids)

    distribution = Counter(
        chunks_per_document.values()
    )

    one_chunk_documents = distribution.get(1, 0)
    two_chunk_documents = distribution.get(2, 0)
    three_plus_documents = sum(
        count
        for chunk_count, count in distribution.items()
        if chunk_count >= 3
    )

    # --------------------------------------------------------
    # STEP 5 — IDENTIFY VERY SMALL CHUNKS
    # --------------------------------------------------------

    tiny_threshold = 50
    small_threshold = 100

    tiny_chunks = [
        chunk
        for chunk in chunks
        if chunk["metadata"]["token_count"] < tiny_threshold
    ]

    small_chunks = [
        chunk
        for chunk in chunks
        if chunk["metadata"]["token_count"] < small_threshold
    ]

    # --------------------------------------------------------
    # STEP 6 — IDENTIFY THE SHORTEST CHUNKS
    # --------------------------------------------------------

    sorted_chunks = sorted(
        chunks,
        key=lambda x: x["metadata"]["token_count"]
    )

    shortest_chunks = []

    for chunk in sorted_chunks[:10]:

        shortest_chunks.append({
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "chunk_number": chunk["metadata"]["chunk_number"],
            "token_count": chunk["metadata"]["token_count"],
            "character_count": len(chunk["text"]),
            "text_preview": chunk["text"][:200]
                .replace("\n", " ")
        })

    # --------------------------------------------------------
    # STEP 7 — CHECK DUPLICATE CHUNK TEXT
    # --------------------------------------------------------

    text_counter = Counter(
        chunk["text"]
        for chunk in chunks
    )

    duplicate_text_groups = [
        {
            "text_preview": text[:200].replace("\n", " "),
            "occurrences": count
        }
        for text, count in text_counter.items()
        if count > 1
    ]

    # --------------------------------------------------------
    # STEP 8 — CHECK CHUNK NUMBERING
    # --------------------------------------------------------

    numbering_issues = []

    chunks_by_document = defaultdict(list)

    for chunk in chunks:
        chunks_by_document[
            chunk["document_id"]
        ].append(chunk)

    for document_id, document_chunks in chunks_by_document.items():

        document_chunks.sort(
            key=lambda x: x["metadata"]["chunk_number"]
        )

        expected_numbers = list(
            range(1, len(document_chunks) + 1)
        )

        actual_numbers = [
            chunk["metadata"]["chunk_number"]
            for chunk in document_chunks
        ]

        if actual_numbers != expected_numbers:
            numbering_issues.append({
                "document_id": document_id,
                "expected": expected_numbers,
                "actual": actual_numbers
            })

    if numbering_issues:
        raise ValueError(
            f"Chunk numbering issues found in "
            f"{len(numbering_issues)} documents."
        )

    print("[SUCCESS] Chunk numbering is consistent.")

    # --------------------------------------------------------
    # STEP 9 — CHECK TOKEN BOUNDARIES
    # --------------------------------------------------------

    boundary_issues = []

    for chunk in chunks:

        metadata = chunk["metadata"]

        start_token = metadata.get("start_token")
        end_token = metadata.get("end_token")
        token_count = metadata.get("token_count")

        if start_token is None or end_token is None:
            boundary_issues.append({
                "chunk_id": chunk["chunk_id"],
                "issue": "Missing token boundary"
            })
            continue

        if end_token <= start_token:
            boundary_issues.append({
                "chunk_id": chunk["chunk_id"],
                "issue": "Invalid token boundary"
            })
            continue

        if (end_token - start_token) != token_count:
            boundary_issues.append({
                "chunk_id": chunk["chunk_id"],
                "issue": "Boundary/token count mismatch"
            })

    if boundary_issues:
        raise ValueError(
            f"Found {len(boundary_issues)} token boundary issues."
        )

    print("[SUCCESS] Token boundaries are consistent.")

    # --------------------------------------------------------
    # STEP 10 — CHECK SLIDING-WINDOW OVERLAP
    # --------------------------------------------------------

    overlap_issues = []

    for document_id, document_chunks in chunks_by_document.items():

        document_chunks.sort(
            key=lambda x: x["metadata"]["chunk_number"]
        )

        for i in range(len(document_chunks) - 1):

            current = document_chunks[i]
            following = document_chunks[i + 1]

            current_end = current["metadata"]["end_token"]
            next_start = following["metadata"]["start_token"]

            actual_gap = next_start - current_end

            # For non-final chunks, expected movement is:
            # next_start = current_start + 450
            #
            # Therefore:
            # next_start - current_end = -50
            #
            # Negative 50 means 50-token overlap.

            if actual_gap != -CHUNK_OVERLAP:
                overlap_issues.append({
                    "document_id": document_id,
                    "current_chunk": current["chunk_id"],
                    "next_chunk": following["chunk_id"],
                    "expected_gap": -CHUNK_OVERLAP,
                    "actual_gap": actual_gap
                })

    if overlap_issues:
        raise ValueError(
            f"Found {len(overlap_issues)} overlap issues."
        )

    print("[SUCCESS] Token overlap is consistent.")

    # --------------------------------------------------------
    # STEP 11 — CHECK DOCUMENT COVERAGE
    # --------------------------------------------------------

    unique_documents = set(document_ids)

    if len(unique_documents) != EXPECTED_DOCUMENTS:
        raise ValueError(
            f"Expected {EXPECTED_DOCUMENTS} unique documents, "
            f"found {len(unique_documents)}."
        )

    print(
        f"[SUCCESS] All {EXPECTED_DOCUMENTS} source documents "
        f"are represented."
    )

    # --------------------------------------------------------
    # STEP 12 — SUMMARY STATISTICS
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("CHUNK QUALITY RESULTS")
    print("-" * 60)

    print(f"Documents represented     : {len(unique_documents)}")
    print(f"Total chunks              : {len(chunks)}")

    print()
    print("TOKEN DISTRIBUTION")
    print(f"Minimum                   : {min(token_counts)}")
    print(f"Maximum                   : {max(token_counts)}")
    print(f"Mean                      : {mean(token_counts):.2f}")
    print(f"Median                    : {median(token_counts):.0f}")

    print()
    print("CHARACTER DISTRIBUTION")
    print(f"Minimum                   : {min(character_counts)}")
    print(f"Maximum                   : {max(character_counts)}")
    print(f"Mean                      : {mean(character_counts):.2f}")
    print(f"Median                    : {median(character_counts):.0f}")

    print()
    print("DOCUMENT → CHUNK DISTRIBUTION")
    print(f"1 chunk                   : {one_chunk_documents}")
    print(f"2 chunks                  : {two_chunk_documents}")
    print(f"3+ chunks                 : {three_plus_documents}")

    print()
    print("SMALL CHUNK ANALYSIS")
    print(
        f"< {tiny_threshold} tokens          : "
        f"{len(tiny_chunks)}"
    )
    print(
        f"< {small_threshold} tokens         : "
        f"{len(small_chunks)}"
    )

    print()
    print("DUPLICATE TEXT")
    print(
        f"Duplicate text groups     : "
        f"{len(duplicate_text_groups)}"
    )

    # --------------------------------------------------------
    # STEP 13 — DISPLAY SHORTEST CHUNKS
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("SHORTEST 10 CHUNKS")
    print("-" * 60)

    for item in shortest_chunks:

        print(
            f"{item['chunk_id']} | "
            f"{item['token_count']} tokens | "
            f"{item['character_count']} chars | "
            f"{item['text_preview']}..."
        )

    # --------------------------------------------------------
    # STEP 14 — CREATE PERSISTED ANALYSIS
    # --------------------------------------------------------

    analysis = {
        "stage": "chunk_quality_analysis",
        "status": "success",

        "configuration": {
            "chunk_size_tokens": CHUNK_SIZE,
            "chunk_overlap_tokens": CHUNK_OVERLAP,
            "expected_documents": EXPECTED_DOCUMENTS,
            "expected_chunks": EXPECTED_CHUNKS
        },

        "corpus": {
            "documents_represented": len(unique_documents),
            "total_chunks": len(chunks)
        },

        "token_statistics": {
            "minimum": min(token_counts),
            "maximum": max(token_counts),
            "mean": mean(token_counts),
            "median": median(token_counts)
        },

        "character_statistics": {
            "minimum": min(character_counts),
            "maximum": max(character_counts),
            "mean": mean(character_counts),
            "median": median(character_counts)
        },

        "document_chunk_distribution": {
            "one_chunk_documents": one_chunk_documents,
            "two_chunk_documents": two_chunk_documents,
            "three_plus_chunk_documents": three_plus_documents
        },

        "small_chunk_analysis": {
            "tiny_threshold_tokens": tiny_threshold,
            "small_threshold_tokens": small_threshold,
            "tiny_chunks_count": len(tiny_chunks),
            "small_chunks_count": len(small_chunks)
        },

        "duplicate_text_groups": len(
            duplicate_text_groups
        ),

        "integrity_checks": {
            "unique_chunk_ids": True,
            "all_documents_represented": True,
            "token_boundaries_valid": True,
            "token_counts_within_limit": True,
            "overlap_consistent": True,
            "chunk_numbering_consistent": True
        },

        "shortest_chunks": shortest_chunks,

        "paths": {
            "source_chunks": str(CHUNKS_PATH),
            "analysis_checkpoint": str(ANALYSIS_PATH)
        }
    }

    # --------------------------------------------------------
    # STEP 15 — SAVE ANALYSIS
    # --------------------------------------------------------

    atomic_json_save(
        analysis,
        ANALYSIS_PATH
    )

    print()
    print("[SUCCESS] Chunk quality analysis checkpoint saved.")
    print(f"[INFO] Path: {ANALYSIS_PATH}")

    print()
    print("=" * 60)
    print("CHUNK QUALITY ANALYSIS COMPLETE")
    print("=" * 60)
    print("[SUCCESS] No chunks were modified.")
    print("[SUCCESS] No source documents were modified.")
    print("[SUCCESS] Integrity checks passed.")

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    del chunks
    del token_counts
    del character_counts
    del chunks_per_document
    del distribution
    del tiny_chunks
    del small_chunks
    del sorted_chunks
    del shortest_chunks
    del text_counter
    del duplicate_text_groups
    del chunks_by_document
    del analysis

    gc.collect()

    print("[INFO] Temporary analysis objects released.")

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] CHUNK QUALITY ANALYSIS FAILED")
    print("=" * 60)
    print(f"Error type    : {type(e).__name__}")
    print(f"Error message : {e}")
    print()
    print("[INFO] Chunk data was not modified.")
    print("[INFO] No invalid analysis checkpoint was accepted.")
    raise

CHUNK QUALITY & INTEGRITY ANALYSIS
[SUCCESS] Loaded 164 persisted chunks.
[SUCCESS] Structural validation passed.
[SUCCESS] Chunk IDs are unique.
[SUCCESS] Chunk numbering is consistent.
[SUCCESS] Token boundaries are consistent.
[SUCCESS] Token overlap is consistent.
[SUCCESS] All 100 source documents are represented.

------------------------------------------------------------
CHUNK QUALITY RESULTS
------------------------------------------------------------
Documents represented     : 100
Total chunks              : 164

TOKEN DISTRIBUTION
Minimum                   : 53
Maximum                   : 500
Mean                      : 347.00
Median                    : 474

CHARACTER DISTRIBUTION
Minimum                   : 292
Maximum                   : 2701
Mean                      : 1670.66
Median                    : 2196

DOCUMENT → CHUNK DISTRIBUTION
1 chunk                   : 36
2 chunks                  : 64
3+ chunks                 : 0

SMALL CHUNK ANALYSIS
< 50 tokens      